限制模型调用次数，避免无限循环，控制调用成本。

初始化模型

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

main_model = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)


举例1：整个会话限制-优雅退出（end）和抛异常（error）

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.runnables import RunnableConfig
from typing import List

agent = create_agent(
    model=main_model,
    checkpointer=InMemorySaver(),  # 在Thread Limiting 中需要
    tools=[],
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=2, #每个线程最多2次模型调用
          #  run_limit=5, #每次运行最多5次
            exit_behavior="error", # end 达到限制后退出， error 整个会话限制-抛异常
        ),
    ],
)

def pretty_iterate_msg(messages: List[SystemMessage | HumanMessage | AIMessage | ToolMessage]):
    for msg in messages:
        msg.pretty_print()

my_config: RunnableConfig = {
    "configurable": {
        "thread_id": "1"
    }
}

response_first = agent.invoke(
    {"messages": [HumanMessage("你好")],},
    config=my_config,
)
print("=" * 30, "> first <", "=" * 30 )
pretty_iterate_msg(response_first["messages"])

response_second = agent.invoke(
    {"messages": [HumanMessage("你是谁？")],},
    config=my_config,
)
print("=" * 30, "> second <", "=" * 30 )
pretty_iterate_msg(response_second["messages"])

response_third = agent.invoke(
    {"messages": [HumanMessage("你能帮我做什么？")],},
    config=my_config,
)
print("=" * 30, "> third <", "=" * 30 )
pretty_iterate_msg(response_third["messages"])

============================== > first < ==============================
================================ Human Message =================================

你好
================================== Ai Message ==================================

你好！有什么我可以帮你的吗？
============================== > second < ==============================
================================ Human Message =================================

你好
================================== Ai Message ==================================

你好！有什么我可以帮你的吗？
================================ Human Message =================================

你是谁？
================================== Ai Message ==================================

我是 Qwen（通义千问），由阿里巴巴集团旗下通义实验室自主研发的大语言模型。有什么我可以帮你的吗？


ModelCallLimitExceededError: Model call limits exceeded: thread limit (2/2)

举例3：单次调用限制-优雅退出（抛异常）

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langchain_core.runnables import RunnableConfig

from pydantic import BaseModel, Field, SecretStr
from typing import Union
from dotenv import load_dotenv
import os

load_dotenv(override=True)

model = ChatDeepSeek(
    model="any",
    base_url="http://127.0.0.1:8889",
    api_key=SecretStr(os.getenv('DEEPSEEK_API_KEY'))
)

class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")

class EventInfo(BaseModel):
    event_name: str = Field(description="事件名称")
    data: str = Field(description="事件发生日期")

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    tools=[],
    middleware=[
        ModelCallLimitMiddleware(
           # thread_limit=2,
            run_limit=3,
            exit_behavior="end",
        )
    ],
    response_format=Union[ContactInfo, EventInfo],
)

config: RunnableConfig  = {
     "configurable": {
        "thread_id": "1"
    }
}

response = agent.invoke(
    {
        "messages": [HumanMessage("你好")],
    },
    config=config
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_1)
 Call ID: call_1
  Args:
    name: 小伙子
    email: xiaohuozi@qq.com
    phone: 12345678912
  EventInfo (call_2)
 Call ID: call_2
  Args:
    event_name: 问数项目启动会
    date: 2026-03-27
================================= Tool Message =================================
Name: ContactInfo

Error: Model incorrectly returned multiple structured responses (ContactInfo, EventInfo) when only one is expected.
 Please fix your mistakes.
================================= Tool Message =================================
Name: EventInfo

Error: Model incorrectly returned multiple structured responses (ContactInfo, EventInfo) when only one is expected.
 Please fix your mistakes.
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_1)
 Call ID: call_1
 